In [ ]:
%pip install --upgrade --force-reinstall torch --index-url https://download.pytorch.org/whl/cpu
import torch.nn as nn

Microsoft Visual C++ Redistributable is not installed, this may lead to the DLL load failure.
It can be downloaded at https://aka.ms/vs/17/release/vc_redist.x64.exe


OSError: [WinError 126] The specified module could not be found. Error loading "C:\Users\sharm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super(TransformerEncoderBlock, self).__init__()
        
        # 1. Multi-Head Attention component (using PyTorch's built-in optimized module)
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True)
        
        # 2. Position-wise Feed-Forward Network
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        # 3. Layer Normalization for both sub-layer outputs
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # 4. Dropout for regularization
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)
            mask: Optional attention mask
        """
        # --- Sub-layer 1: Self-Attention + Residual & Norm ---
        # MultiheadAttention returns (attn_output, attn_weights)
        attn_out, _ = self.self_attn(x, x, x, attn_mask=mask)
        # Residual connection + dropout + layer normalization
        x = self.norm1(x + self.dropout1(attn_out))
        
        # --- Sub-layer 2: Feed-Forward + Residual & Norm ---
        ff_out = self.feed_forward(x)
        # Residual connection + dropout + layer normalization
        x = self.norm2(x + self.dropout2(ff_out))
        
        return x

# --- Example Usage ---
if __name__ == "__main__":
    batch_size = 2
    seq_len = 10
    d_model = 256
    num_heads = 4
    d_ff = 512
    
    # Initialize the encoder block
    encoder_block = TransformerEncoderBlock(d_model=d_model, num_heads=num_heads, d_ff=d_ff)
    
    # Simulate an input token embedding sequence
    tokens = torch.randn(batch_size, seq_len, d_model)
    
    # Pass through the transformer encoder block
    output = encoder_block(tokens)
    print("Transformer Encoder Block Output Shape:", output.shape)

Transformer Encoder Block Output Shape: torch.Size([2, 10, 256])
